# Synthetic MMM exploration

Uses observable synthetic data only. Weeks 1–100 train and weeks 101–120 hold out. Holdout spend and controls are realized values; predictions using them are conditional.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from mmm_lab.paths import RAW_DATA_DIR, PROCESSED_DATA_DIR
raw = pd.read_csv(RAW_DATA_DIR / "synthetic_mmm.csv", parse_dates=["week_start"])
ready = pd.read_csv(PROCESSED_DATA_DIR / "model_ready.csv")
assert len(raw) == 720
assert ready.split.value_counts().to_dict() == {"train": 600, "holdout": 120}
boundary = raw.loc[raw.week_index == 101, "week_start"].iloc[0]
spends = ["coupon_spend", "retailer_spend", "digital_spend"]
raw.head()

## Sales trends by segment

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
for segment, group in raw.groupby("segment"):
    ax.plot(group.week_start, group.sales_units, label=segment)
ax.axvline(boundary, color="black", linestyle="--", label="Holdout")
ax.set(xlabel="Week", ylabel="Sales units", title="Weekly sales by segment")
ax.legend(ncol=7)
plt.show()

## Marketing spend over time

In [ ]:
weekly_spend = raw.groupby("week_start")[spends].sum()
fig, ax = plt.subplots(figsize=(11, 4))
for channel in spends:
    ax.plot(weekly_spend.index, weekly_spend[channel], label=channel)
ax.axvline(boundary, color="black", linestyle="--")
ax.set(xlabel="Week", ylabel="Dollars per week", title="Total channel spend")
ax.legend()
plt.show()

## Seasonality

Descriptive averages also contain trend, holidays, and promotional effects.

In [ ]:
cycle = raw.assign(cycle_week=(raw.week_index - 1) % 52 + 1)
ax = cycle.groupby("cycle_week").sales_units.mean().plot(figsize=(10, 4))
ax.set(xlabel="Week of annual cycle", ylabel="Mean sales units", title="Observed seasonal profile")
plt.show()

## Sales distribution

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for segment, group in raw.groupby("segment"):
    ax.hist(group.sales_units, bins=18, alpha=0.4, label=segment)
ax.set(xlabel="Sales units", ylabel="Observations", title="Weekly sales distribution")
ax.legend()
plt.show()

## Channel spend correlations

Correlation is not an identified effect.

In [ ]:
corr = raw[spends].corr()
fig, ax = plt.subplots(figsize=(5, 4))
image = ax.imshow(corr, vmin=-1, vmax=1, cmap="coolwarm")
ax.set_xticks(range(3), spends, rotation=30)
ax.set_yticks(range(3), spends)
for i in range(3):
    for j in range(3):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center")
fig.colorbar(image, ax=ax)
ax.set_title("Channel spend correlation")
plt.show()

## Segment differences

The smaller segments have weaker promotional information.

In [ ]:
summary = raw.assign(total_spend=raw[spends].sum(axis=1)).groupby("segment")[["sales_units", "total_spend"]].mean()
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
summary.sales_units.plot.bar(ax=axes[0], title="Mean weekly sales", ylabel="Units")
summary.total_spend.plot.bar(ax=axes[1], title="Mean weekly spend", ylabel="Dollars")
fig.tight_layout()
plt.show()

## Training and holdout boundary

Pre-holdout spend history must remain available for adstock.

In [ ]:
weekly = raw.groupby("week_start").sales_units.mean()
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(weekly.index, weekly)
ax.axvspan(boundary, weekly.index.max(), color="tab:orange", alpha=0.2, label="Holdout")
ax.axvline(boundary, color="black", linestyle="--")
ax.set(xlabel="Week", ylabel="Mean sales units", title="Temporal split")
ax.legend()
plt.show()